# Гистограмма и преобразование интенсивностей пикселов полутоновых изображений

Цель: Освоить способы трансформации изображения спомощьюпопиксельной обработки, научиться использовать гистограммыизображений

### Задание 1

In [ ]:
from PIL import Image
from PIL import ImageDraw
from PIL import ImageOps
import matplotlib.pyplot as plt
from matplotlib.pyplot import hist
import numpy as np

%matplotlib inline

In [ ]:
# открываем картинку и переводим в grayscale-режим
easy_img = Image.open("Простой рисунок.png").convert('L').convert('RGB')
coins_img = Image.open("Coins.jpg").convert('RGB')
rice_img = Image.open("rice.jpg").convert('RGB')
arch_img = Image.open("arch.jpg").convert('RGB')

In [ ]:
def draw_image(image):
    plt.figure(figsize=(20,13), dpi=90)
    plt.subplot2grid((3,3), (0,0))
    plt.imshow(image,  cmap='gray')
    plt.title('Исходное изображение')

    #Гистограмма
    plt.subplot2grid((3,3), (0,1))
    plt.hist(np.ravel(image), bins=256)
    plt.title('Гистограмма интенсивности цвета')

#### Гистограммы изображений

In [ ]:
draw_image(easy_img)

По гистограмме видно, что на изображении соседние пиксели имеют одинаковую интенсивность.
Так же видно 3 пика:
    - самый правый пик, почти белый цвет, имеет более 80000 пикселей, это фон изображения
    - средний пик, около 60000 пикселей - сетка
    - пик левее, около 10000 пикселей - стол, стул и тд.

In [ ]:
draw_image(coins_img)

По гистограмме видно, что изображение с оттенками серого.
Есть основное пик, с большим количеством пикселей - фон.
Остальные пики это монеты.

In [ ]:
draw_image(rice_img)

3 основных пика.
Крайний левый - темный фон.
Средний - фон светлее.
Правый пик - рис.

In [ ]:
draw_image(arch_img)

Изображение с низкой контрастностью, пики не ярко выражены, за исключением праого столбца, это небо.

#### Кусочно-линейные преобразования

In [ ]:
# функция для отображения оригинального и преобразованного изображений, а также их гистограмм
def draw_image3(image, imageNew):
    plt.figure(figsize=(16,13), dpi=90)
    plt.subplot2grid((3,3), (0,0))
    plt.imshow(image, cmap='gray')
    plt.title('Исходное изображение')
    plt.subplot2grid((3,3), (0,1))
    plt.imshow(imageNew, cmap='gray')
    plt.title('Преобразованное изображение')

    #Гистограммы двух изображений
    plt.subplot2grid((3,3), (0,2))
    plt.hist(np.ravel(image), bins=256, label=['Оригинал'])
    plt.hist(np.ravel(imageNew), bins=256, label=['Результат'])
    plt.title('Гистограмма интенсивности цвета')
    plt.legend()
    plt.show()

In [ ]:
# функция для выполнения преобразований с заданной функцией преобразования f
def transform(f, img):
  img_copy = img.copy() #преобразовывать будем копию исходного изображения
  draw = ImageDraw.Draw(img_copy) # создаем инструмент для рисования
  width = img_copy.size[0] # определяем ширину
  height = img_copy.size[1] # определяем высоту
  pixels = img_copy.load() # выгружаем значения пикселей
  for i in range(width):
    for j in range(height):
      r, g, b = pixels[i, j]
      S = tuple(map(f, (r, g, b)))
      draw.point((i, j), S)

  draw_image3(img, img_copy)

In [ ]:
def f1(x): # 1е из кусочно-линейных преобразований
  x1 = 70
  x2 = 210
  if x < x1 or x > x2:
    return int(2 * x)
  return int(0.5 * x)

def f2(x): # 2е из кусочно-линейных преобразований
  x1 = 100
  x2 = 200
  if x < x1 or x > x2:
    return int(0.5 * x)
  return int(2 * x)

In [ ]:
transform(f1, easy_img)

Функция f1 сдвигает вправо интенсивность пикселей меньше 70 или больше 210 в 2 раза, что видно на гистограмме.
Пиксели в районе 150 наоборот свдигаются влево в 2 раза.

In [ ]:
transform(f2, easy_img)

Функция f2 наоборот, сдвигает "средние" пиксели вправо, а крайние влево.

In [ ]:
transform(f1, coins_img)

In [ ]:
transform(f2, coins_img)

Сдвиг фона в 2 раза, уперлись в 255 -> потеряли информацию.

In [ ]:
transform(f1, rice_img)

In [ ]:
transform(f2, rice_img)

In [ ]:
transform(f1, arch_img)

In [ ]:
transform(f2, arch_img)

In [ ]:
def f3(x): # интервальная бинаризация
  x1 = 50
  x2 = 150
  if x2 > x > x1:
    return 255
  return 0


def f4(x): # интервальная бинаризация
  x2 = 130
  if x < x2:
    return 0
  return 255

In [ ]:
transform(f3, easy_img)

Пиксели от 150 сделали черными

In [ ]:
transform(f4, easy_img)

In [ ]:
transform(f3, coins_img)

In [ ]:
transform(f4, coins_img)

In [ ]:
transform(f3, rice_img)

In [ ]:
transform(f4, rice_img)

In [ ]:
transform(f3, arch_img)

In [ ]:
transform(f4, arch_img)

In [ ]:
def f5(x):
  x1 = 50
  x2 = 150
  if x2 > x > x1:
    return 255
  return x # не меняем

In [ ]:
transform(f5, easy_img)

In [ ]:
transform(f5, coins_img)

In [ ]:
transform(f5, rice_img)

In [ ]:
transform(f5, arch_img)

#### Пороговая сегментация для многопиковых гистограмм

In [ ]:
def transform(f, limit, img):
  img_copy = img.copy()
  draw = ImageDraw.Draw(img_copy)
  width = img_copy.size[0]
  height = img_copy.size[1]
  pixels = img_copy.load()
  for i in range(width):
    for j in range(height):
      r, g, b = pixels[i, j]
      S = tuple(map(f, (r, g, b), (limit, limit, limit)))
      draw.point((i, j), S)

  draw_image3(img, img_copy)

In [ ]:
def f_bin(x, limit):
  return 0 if x in range(*limit) else 255

In [ ]:
transform(f_bin, (200, 220), easy_img)

Колба лампы находилась в диапазоне 200, 220, сделали ее черной.

In [ ]:
transform(f_bin, (45, 60), coins_img)

In [ ]:
transform(f_bin, (180, 250), rice_img)

In [ ]:
transform(f_bin, (100, 130), arch_img)

Вывод: с помощью данных преобразований и настроки порогов можно выделять необходимые объекты.
Анализ гистограмм позволяет понять какая интесивность пикселей преобладает на картинке.

### Задание 2

#### Фильтрация изображений

In [ ]:
import cv2 # импорт Open CV

In [ ]:
light_img = cv2.imread('Light.jpg')
camera_img = cv2.imread('Camera.jpg')
building_img = cv2.imread('Building.jpg')
truck_img = cv2.imread('Truck.jpg')
statue_img = cv2.imread('Statue.jpg')
lena_img = cv2.imread('Noisy Lena.jpg')
bri_img = cv2.imread('Bri.jpg')

In [ ]:
def draw_image_2(image, imageNew, n):
    plt.figure(figsize=(16,13), dpi=100)
    plt.subplot2grid((3,3), (0,0)) #1 - исходное изображение
    plt.title('Исходное изображение')
    plt.imshow(image) #Вывод
    plt.subplot2grid((3,3), (0,1)) #2 - преобразованное изображение
    plt.title('Преобразованное изображение')
    plt.imshow(imageNew) #Вывод

    #Гистограммы двух изображений
    plt.subplot2grid((3,3), (0,2))
    plt.hist(np.ravel(image), bins=256, label=['Оригинал'])
    plt.hist(np.ravel(imageNew), bins=256, label=['Результат'])
    plt.legend()
    plt.title('n = %d' % n)
    plt.show()

In [ ]:
def my_filter(filter_fun, image):
  parametrs = [3,5,7,9] # значения размера ядра
  # применение фильтра для изображения
  for parametr in parametrs:
    draw_image_2(image, filter_fun(image, (parametr,parametr),0), parametr)

##### Усредняющий фильтр

In [ ]:
my_filter(cv2.blur, light_img)
my_filter(cv2.blur, camera_img)
my_filter(cv2.blur, building_img)
my_filter(cv2.blur, statue_img)
my_filter(cv2.blur, lena_img)
my_filter(cv2.blur, bri_img)

##### Фильтр Гаусса

In [ ]:
my_filter(cv2.GaussianBlur, light_img)
my_filter(cv2.GaussianBlur, camera_img)
my_filter(cv2.GaussianBlur, building_img)
my_filter(cv2.GaussianBlur, statue_img)
my_filter(cv2.GaussianBlur, lena_img)
my_filter(cv2.GaussianBlur, bri_img)

##### Медианный фильтр

In [ ]:
def median_filter(image):
  parametrs = [3,5,7,9] # значения размера ядра
  # применение фильтра для изображения
  for parametr in parametrs:
    draw_image_2(image, cv2.medianBlur(image, parametr), parametr)

In [ ]:
median_filter(light_img)
median_filter(camera_img)
median_filter(building_img)
median_filter(statue_img)
median_filter(lena_img)
median_filter(bri_img)

Усредняющий фильтр сильно размывает границы.
Фильтр Гаусса дает плавное размытие.
Медианный фильтр убирает шум и сохраняет границы.
На гистограмме пики разглаживаются, пики становятся ниже и шире.

##### Фильтр Превитта

In [ ]:
def previtt_filter1(image):
  previtt_mtx = [[-1,0,1],[-1,0,1],[-1,0,1]] # ядро фильтра Превитта
  kernel = np.array(previtt_mtx, np.float32) # приведение ядра фильтра к типу float
  draw_image_2(image, cv2.filter2D(image, -1, kernel), 3)

In [ ]:
previtt_filter1(light_img)
previtt_filter1(camera_img)
previtt_filter1(building_img)
previtt_filter1(statue_img)
previtt_filter1(lena_img)
previtt_filter1(bri_img)

In [ ]:
# трансонируем фильтр
def previtt_filter2(image):
  previtt_mtx = [[-1,0,1],[-1,0,1],[-1,0,1]] # ядро фильтра Превитта
  previtt_mtx = np.array(previtt_mtx).T
  kernel = np.array(previtt_mtx, np.float32) # приведение ядра фильтра к типу float
  draw_image_2(image, cv2.filter2D(image, -1, kernel), 3)

In [ ]:
previtt_filter2(light_img)
previtt_filter2(camera_img)
previtt_filter2(building_img)
previtt_filter2(statue_img)
previtt_filter2(lena_img)
previtt_filter2(bri_img)

##### Фильтр Собеля

In [ ]:
def sobel_filter1(image):
  sobel_mtx = [[-1,0,1],[-2,0,2],[-1,0,1]] # ядро фильтра Собеля
  kernel = np.array(sobel_mtx, np.float32) # приведение ядра фильтра к типу float
  draw_image_2(image, cv2.filter2D(image, -1, kernel), 3)

In [ ]:
sobel_filter1(light_img)
sobel_filter1(camera_img)
sobel_filter1(building_img)
sobel_filter1(statue_img)
sobel_filter1(lena_img)
sobel_filter1(bri_img)

In [ ]:
# трансонируем фильтр
def sobel_filter2(image):
  sobel_mtx = [[-1,0,1],[-2,0,2],[-1,0,1]] # ядро фильтра Собеля
  sobel_mtx = np.array(sobel_mtx).T
  kernel = np.array(sobel_mtx, np.float32) # приведение ядра фильтра к типу float
  draw_image_2(image, cv2.filter2D(image, -1, kernel), 3)

##### Фильтр Робертса

In [ ]:
def roberts_filter(image):
  roberts1_mtx = [[0,1],[-1,0]] # 1 ядро фильтра Робертса
  roberts2_mtx = [[1,0],[0,-1]] # 2 ядро фильтра Робертса
  roberts1_mtx = np.array(roberts1_mtx, np.float32)
  roberts2_mtx = np.array(roberts2_mtx, np.float32)
  g1 = cv2.filter2D(image, cv2.CV_64F, roberts1_mtx, borderType=cv2.BORDER_REPLICATE)
  g2 = cv2.filter2D(image, cv2.CV_64F, roberts2_mtx, borderType=cv2.BORDER_REPLICATE)
  magnitude = np.sqrt(g1**2 + g2**2)
  magnitude = np.clip(magnitude, 0, 255).astype(np.uint8)
  draw_image_2(image, magnitude, 3)

In [ ]:
roberts_filter(light_img)
roberts_filter(camera_img)
roberts_filter(building_img)
roberts_filter(statue_img)
roberts_filter(lena_img)
roberts_filter(bri_img)

Дифференциальные фильтры выделяют границы объекта, делая их светлее.
У Превитта границы жирнее, у Собеля четче, Робертс самый шумный.
Гистограммы сильно отличаются от оригинала, виден пик у нуля.